### 0. 라이브러리 및 데이터 불러오기

In [2]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

engine = create_engine('mysql+pymysql://root:@localhost/review_analysis?charset=utf8mb4')

df_pd = pd.read_sql("SELECT * FROM products_all", engine)

query_reviews = """
SELECT * FROM reviews_all
WHERE STR_TO_DATE(작성일, '%%Y-%%m-%%d') <= '2026-03-31'
"""
df_rv = pd.read_sql(query_reviews, engine)

print(f"products: {df_pd.shape}")
print(f"reviews:  {df_rv.shape}")


products: (1541, 12)
reviews:  (548248, 14)


### 1. 작성일: str -> datetime 변환

In [6]:
df_rv['작성일'] = pd.to_datetime(df_rv['작성일'])
print(f"1. 작성일 변환 완료: {df_rv['작성일'].dtype}")

1. 작성일 변환 완료: datetime64[us, UTC+09:00]


### 2. 체험단/사진유무: str -> boolean 변환
- 'TRUE'/'FALSE'/'True'/'False' 혼재 → boolean

In [8]:
for col in ['체험단', '사진유무']:
    df_rv[col] = (
        df_rv[col]
        .astype(str)
        .str.strip()
        .str.upper()
        .map({'TRUE': True, 'FALSE': False})
        .astype('boolean')
    )

print(f"2. 체험단/사진유무 boolean 변환 완료")
print(f"   체험단: {df_rv['체험단'].value_counts().to_dict()}")
print(f"   사진유무: {df_rv['사진유무'].value_counts().to_dict()}")


2. 체험단/사진유무 boolean 변환 완료
   체험단: {np.False_: 547886, np.True_: 362}
   사진유무: {np.False_: 347849, np.True_: 200399}


### 3. 평점: 0점 -> NaN 처리
- 0점은 무효 리뷰 (2024년 8~10월에 집중 발생)
- 원본 보존용 평점_raw 컬럼 생성

In [10]:
df_rv['평점_raw'] = df_rv['평점']
df_rv['평점'] = df_rv['평점'].replace(0, np.nan)

print(f"3. 평점 0점 NaN 처리 완료")
print(f"   0점 처리 건수: {(df_rv['평점_raw'] == 0).sum()}건 → NaN")

3. 평점 0점 NaN 처리 완료
   0점 처리 건수: 0건 → NaN


### 4. 만족도: 텍스트 파싱 → 개별 컬럼 분리 + 순서형 범주형 변환
- 예: "사이즈: 정사이즈 / 퀄리티: 좋음" → 사이즈=정사이즈, 퀄리티=좋음

In [11]:
def _parse_satisfaction(text):
    if pd.isna(text) or text == "":
        return None
    result = {}
    for part in str(text).split('/'):
        if ':' in part:
            key, value = part.split(':', 1)
            result[key.strip()] = value.strip()
    return result

parsed = df_rv['만족도'].apply(_parse_satisfaction)
sat_df = pd.DataFrame(parsed.dropna().tolist())
sat_df.index = df_rv[df_rv['만족도'].notnull()].index
sat_df = sat_df.reindex(df_rv.index)

# 응답여부 플래그
df_rv['만족도_응답여부'] = np.where(df_rv['만족도'].notnull(), '응답', '미응답')

# 개별 항목 컬럼 추가
for col in sat_df.columns:
    df_rv[col] = sat_df[col]

# 순서형 범주형(Ordered Categorical) 변환
categories_dict = {
    '사이즈':       ['매우 작음', '조금 작음', '정사이즈', '조금 큼', '많이 큼'],
    '화면 대비 색감': ['매우 어두움', '어두움', '화면과 비슷', '밝음', '매우 밝음'],
    '퀄리티':       ['매우 나쁨', '나쁨', '보통', '좋음', '매우 좋음'],
    '구김':         ['전혀 없음', '거의 없음', '약간 있음', '많음', '매우 많음'],
    '두께감':       ['매우 얇음', '얇음', '적당함', '두꺼움', '매우 두꺼움'],
    '신축성':       ['전혀 없음', '거의 없음', '적당함', '강함', '매우 강함'],
    '색감':         ['어두움', '화면과 비슷', '밝음', '매우 밝음'],
    '보온성':       ['전혀 없음', '거의 없음', '적당함', '좋음', '매우 좋음'],
}

for col, order in categories_dict.items():
    if col in df_rv.columns:
        df_rv[col] = pd.Categorical(df_rv[col], categories=order, ordered=True)

print(f"4. 만족도 파싱 + 범주형 변환 완료")
print(f"   응답: {(df_rv['만족도_응답여부'] == '응답').sum()}건 / 미응답: {(df_rv['만족도_응답여부'] == '미응답').sum()}건")


4. 만족도 파싱 + 범주형 변환 완료
   응답: 16272건 / 미응답: 531976건


### 5. 도움돼요 파생변수 생성
- 7일 미만 리뷰 제거 (노출 기간 불충분)
- 노출일수: 기준일 - 작성일
- 일평균_도움돼요지수: 도움돼요 / log(1 + 노출일수)
- 도움여부: 도움돼요 > 0 이면 1


In [12]:
기준일 = pd.to_datetime('2026-03-31').normalize().tz_localize(None)

df_rv['노출일수'] = (
    기준일 - df_rv['작성일'].dt.normalize().dt.tz_localize(None)
).dt.days + 1

before = len(df_rv)
df_rv = df_rv[df_rv['노출일수'] >= 7].copy()
after = len(df_rv)

df_rv['일평균_도움돼요지수'] = df_rv['도움돼요'] / np.log1p(df_rv['노출일수'])
df_rv['도움여부'] = (df_rv['도움돼요'] > 0).astype(int)

print(f"5. 도움돼요 파생변수 생성 완료")
print(f"   7일 미만 제거: {before - after}개 ({before}개 → {after}개)")



5. 도움돼요 파생변수 생성 완료
   7일 미만 제거: 440개 (548248개 → 547808개)


### 최종확인

In [13]:
print(f"\n최종 shape: {df_rv.shape}")
print(f"컬럼 목록:\n{df_rv.columns.tolist()}")


최종 shape: (547808, 27)
컬럼 목록:
['리뷰번호', 'goodsNo', '작성자', '리뷰내용', '평점', '체험단', '구매옵션', '키', '몸무게', '성별', '작성일', '만족도', '사진유무', '도움돼요', '평점_raw', '만족도_응답여부', '사이즈', '화면 대비 색감', '퀄리티', '구김', '두께감', '신축성', '색감', '보온성', '노출일수', '일평균_도움돼요지수', '도움여부']
